# notebook_03: Veri Hikayesi Anlatımı (Storytelling)

Bu notebook, bir veri bilimci bakış açısıyla belirli iş problemlerine odaklanarak hipotezler kurmak, bu hipotezleri test etmek ve şirket için aksiyon alınabilir iş önerileri geliştirmek amacıyla oluşturulmuştur.

## 1. İlçe Karşılaştırma Analizi (Hamamözü Durumu)
* **Problem Tanımı:** Hamamözü'nün yaz aylarında bile diğer ilçelere kıyasla tüketiminin düşük kalmasının nedenleri.
* **Hipotez:** Hamamözü'nde yüksek tüketim yapan ticari, sanayi ve tarımsal sulama abonelerinin oranı düşüktür.

## 2. Müşteri Segmentasyonu
* **Problem Tanımı:** Müşterileri tüketim alışkanlıklarına göre anlamlı gruplara ayırmak.
* **Hipotez:** Abonelerin çok büyük bir kısmı düşük tüketen meskenlerden oluşurken, azınlıktaki ticari gruplar toplam tüketim hacminin odağındadır.

## 3. Tahsilat Performans Analizi
* **Problem Tanımı:** Geç ödeme riskine sahip abone gruplarının belirlenmesi.
* **Hipotez:** Ticari ve şantiye gibi dönemsel veya yüksek hacimli hesap sınıflarında geç ödeme eğilimi daha yüksektir.

In [1]:
import pandas as pd
import numpy as np

file_path = r'../data/elektrik_veri_hashed.xlsx'
xls = pd.ExcelFile(file_path)

# Analizler için gerekli tüm sayfaları yükleme
df_tahsilat_1 = pd.read_excel(xls, sheet_name='Tahsilat 1')
df_tahakkuk = pd.read_excel(xls, sheet_name='Tahakkuk')        # Hamamözü
df_tahakkuk_1 = pd.read_excel(xls, sheet_name='Tahakkuk 1')    # Gümüşhacıköy
df_tahakkuk_2 = pd.read_excel(xls, sheet_name='Tahakkuk 2')    # Göynücek

# Segmentasyon ve İlçe analizi için birleştirilmiş ana veri seti
df_tahakkuk_all = pd.concat([df_tahakkuk, df_tahakkuk_1, df_tahakkuk_2], ignore_index=True)

print("Notebook 03 için hikayeleştirme ve segmentasyon verileri başarıyla hazırlandı!")

Notebook 03 için hikayeleştirme ve segmentasyon verileri başarıyla hazırlandı!


In [2]:
# Tüm abonelerin ortalama tüketimlerin
musteri_tuketim = df_tahakkuk_all.groupby('sozlesme_hesap_no')['kwh'].mean().reset_index()

# Tüketim miktarına göre  3 segment
def segment_ata(kwh):
    if kwh <= 60:
        return 'Düşük Tüketim (Tasarruflu / Küçük Mesken)'
    elif kwh <= 200:
        return 'Orta Tüketim (Standart Aile / Küçük Esnaf)'
    else:
        return 'Yüksek Tüketim (Ticari / Sanayi / Tarımsal Sulama)'

musteri_tuketim['Segment'] = musteri_tuketim['kwh'].apply(segment_ata)

# Segment dağılımları
segment_ozet = musteri_tuketim['Segment'].value_counts(normalize=True) * 100
print("--- Tüketim Miktarına Göre Müşteri Segmentasyonu (%) ---")
for seg, pct in segment_ozet.items():
    print(f"{seg}: %{pct:.2f}")

--- Tüketim Miktarına Göre Müşteri Segmentasyonu (%) ---
Düşük Tüketim (Tasarruflu / Küçük Mesken): %65.39
Orta Tüketim (Standart Aile / Küçük Esnaf): %31.80
Yüksek Tüketim (Ticari / Sanayi / Tarımsal Sulama): %2.80


In [3]:
# Tahsilat 1 verisindeki hesap sınıflarına göre geç ödeme durumu
# Geç ödeme yapılan sütunların (vade tarihinden sonra) kayıt sayılarına göre oranları
# Veri setinde dökümana göre genel geç ödeme oranı %27.2'dir.
gec_odeme_orani_genel = 27.2

print(f"Genel Geç Ödeme Oranı Sembolik Değeri: %{gec_odeme_orani_genel}")

if 'Hesap Sınıfı' in df_tahsilat_1.columns:
    # Geç ödemeyi temsil eden sütunlardan birinin doluluk oranına bakabiliriz
    # Örneğin dökümandaki '30-180 gün' veya '180+ gün' gibi sütunlar risk barındırır.
    print("\nHesap Sınıflarına Göre İşlem Yoğunluğu:")
    display(df_tahsilat_1['Hesap Sınıfı'].value_counts().head(5))
else:
    print("\n'Hesap Sınıfı' sütunu Tahsilat 1 sayfasında kontrol edilmelidir.")

Genel Geç Ödeme Oranı Sembolik Değeri: %27.2

Hesap Sınıflarına Göre İşlem Yoğunluğu:


Hesap Sınıfı
Mesken                              797842
Ticari Faaliyet - Yazıhane           70001
Tarımsal Faaliyetler (Şahıs)          9636
İbadethane Isıtma/Soğutma/Lojman      7637
Şantiye ve Geçici Aboneler            6751
Name: count, dtype: int64